In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
from huggingface_hub import login
login()
# Load base model WITHOUT unsloth
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-4b-it",
    torch_dtype=torch.float16,
    device_map="cpu",
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/ARIA/models/gemma-sar-lora"
)

# Load LoRA adapter
print("Loading LoRA...")
model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/ARIA/models/gemma-sar-lora",
)

# Merge LoRA into base
print("Merging...")
model = model.merge_and_unload()

# Save merged model
print("Saving merged model...")
model.save_pretrained("/content/gemma-sar-merged-peft")
tokenizer.save_pretrained("/content/gemma-sar-merged-peft")
print("Merged model saved!")

# Convert to GGUF
!git clone https://github.com/ggerganov/llama.cpp.git /content/llama.cpp 2>/dev/null
!pip install gguf sentencepiece -q

print("Converting to GGUF...")
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/gemma-sar-merged-peft \
    --outfile /content/gemma-sar.gguf \
    --outtype q4_k_m

size = os.path.getsize("/content/gemma-sar.gguf") / 1024/1024
print(f"GGUF: {size:.0f} MB")

!cp /content/gemma-sar.gguf /content/drive/MyDrive/ARIA/models/gemma-sar.gguf
print("✅ DONE — gemma-sar.gguf saved to Drive")

Loading base model...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Loading LoRA...
Merging...
Saving merged model...
